# Dataset Audit

This notebook loads the raw riding dataset, checks schema and label balance, detects session boundaries, estimates sample rate, and saves a cleaned CSV for downstream windowing.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve()
raw_path = project_root / 'data' / 'raw' / 'test.csv'
clean_path = project_root / 'data' / 'processed' / 'clean.csv'
clean_path.parent.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_csv(raw_path, encoding='utf-8-sig')
print('shape=', df.shape)
print('dtypes:')
print(df.dtypes.to_string())
print('\nLabel counts:')
print(df['Label'].value_counts(dropna=False).sort_index())
print('\nGender counts:')
print(df['gender'].value_counts(dropna=False).sort_index())
print('\nTime of day counts:')
print(df['time_of_day'].value_counts(dropna=False).sort_index())

shape= (81606, 14)
dtypes:
Formatted_Timestamp        str
Raw_Timestamp            int64
Latitude               float64
Longitude              float64
Speed                  float64
Ax                     float64
Ay                     float64
Az                     float64
Gx                     float64
Gy                     float64
Gz                     float64
Label                      str
gender                     str
time_of_day                str

Label counts:
Label
BUMP        18030
LEFT         8897
RIGHT        4864
STOP        14735
STRAIGHT    35080
Name: count, dtype: int64

Gender counts:
gender
female     9223
male      72383
Name: count, dtype: int64

Time of day counts:
time_of_day
day      30120
night    51486
Name: count, dtype: int64


In [3]:
df['Formatted_Timestamp'] = pd.to_datetime(df['Formatted_Timestamp'], errors='coerce')
df = df.sort_values(['Raw_Timestamp', 'Formatted_Timestamp'], kind='mergesort').reset_index(drop=True)
df['time_gap_s'] = df['Raw_Timestamp'].diff().fillna(0)
session_ids = (df['time_gap_s'] > 60).cumsum()
df['session_id'] = session_ids
print('Session boundaries (gap > 60s):')
print(df.loc[df['time_gap_s'] > 60, ['Raw_Timestamp', 'Formatted_Timestamp']].head().to_string(index=False))
print('\nSession count =', df['session_id'].nunique())
print('Session sizes:')
print(df.groupby('session_id').size().to_string())

session_rates = []
for sid, group in df.groupby('session_id'):
    timestamps = pd.to_datetime(group['Formatted_Timestamp'])
    diffs = timestamps.diff().dropna().dt.total_seconds()
    if diffs.empty:
        rate_hz = np.nan
    else:
        rate_hz = 1.0 / diffs.median()
    session_rates.append((sid, len(group), rate_hz))
session_rates_df = pd.DataFrame(session_rates, columns=['session_id', 'n_samples', 'estimated_hz'])
print('\nEstimated per-session sample rate (Hz):')
print(session_rates_df.to_string(index=False))

Session boundaries (gap > 60s):
 Raw_Timestamp Formatted_Timestamp
    1764182511 2025-11-27 00:11:51
    1764183720 2025-11-27 00:32:00
    1764184401 2025-11-27 00:43:21
    1764185519 2025-11-27 01:01:59
    1764219631 2025-11-27 10:30:31

Session count = 9
Session sizes:
session_id
0     9223
1    18160
2    11349
3     9557
4    12420
5     6809
6      722
7    12847
8      519



Estimated per-session sample rate (Hz):
 session_id  n_samples  estimated_hz
          0       9223           inf
          1      18160           inf
          2      11349           inf
          3       9557           inf
          4      12420           inf
          5       6809           inf
          6        722           inf
          7      12847           inf
          8        519           inf


C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  rate_hz = 1.0 / diffs.median()
C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  rate_hz = 1.0 / diffs.median()
C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  rate_hz = 1.0 / diffs.median()
C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  rate_hz = 1.0 / diffs.median()
C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  rate_hz = 1.0 / diffs.median()
C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  rate_hz = 1.0 / diffs.median()
C:\Users\BA\AppData\Local\Temp\ipykernel_3868\11200131.py:19: RuntimeW

In [4]:
clean_df = df.copy()
clean_df = clean_df.drop(columns=['time_gap_s', 'session_id'], errors='ignore')
clean_df.to_csv(clean_path, index=False)
print(f'Cleaned data saved to: {clean_path}')

Cleaned data saved to: C:\Users\BA\Desktop\IIIT_TWO_WHEELER\models\two_wheeler_project\data\processed\clean.csv


In [5]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().resolve().parent))
from utils.windowing import make_windows

X, y, meta = make_windows(clean_df, window_size=156, stride=78)
window_path = project_root / 'data' / 'processed' / 'windows.npz'
np.savez(window_path, X=X, y=y, meta=np.array([meta], dtype=object))
print('X shape =', X.shape)
print('y shape =', y.shape)
print('class balance =')
print(pd.Series(y).value_counts().sort_index())
print('meta =', meta)

labels = ['STRAIGHT', 'LEFT', 'RIGHT', 'BUMP', 'STOP']
for label in labels:
    subset = clean_df[clean_df['Label'] == label].head(300)
    if subset.empty:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    for ax, feature in zip(axes, ['Ax', 'Gy']):
        ax.plot(subset[feature].to_numpy(), linewidth=1.2)
        ax.set_title(f'{label} - {feature} (first 300 samples)')
        ax.set_xlabel('Sample index')
        ax.set_ylabel(feature)
    fig.tight_layout()
    plt.show()

X shape = (1045, 156, 7)
y shape = (1045,)
class balance =
BUMP        235
LEFT        115
RIGHT        62
STOP        188
STRAIGHT    445
Name: count, dtype: int64
meta = {'window_size': 156, 'stride': 78, 'feature_cols': ['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz', 'Speed'], 'label_col': 'Label', 'n_windows': 1045, 'class_balance': {'BUMP': 235, 'LEFT': 115, 'RIGHT': 62, 'STOP': 188, 'STRAIGHT': 445}}


C:\Users\BA\AppData\Local\Temp\ipykernel_3868\781376831.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\BA\AppData\Local\Temp\ipykernel_3868\781376831.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\BA\AppData\Local\Temp\ipykernel_3868\781376831.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\BA\AppData\Local\Temp\ipykernel_3868\781376831.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\BA\AppData\Local\Temp\ipykernel_3868\781376831.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
